# CUM Series 8+9 — Generalized Iterate Blending

**Core principle:** NS intermediate iterate blending is the only strategy that beats Muon (40+ experiments). Now we systematically explore:
- Series 8: Different blend pairs, extended NS, three-point blends, SV-space blending
- Series 9: Does the principle generalize beyond NS? (dual-momentum, temporal NS blending)

**Benchmark config:** Same proven setup as M3 CPU benchmark (no overfitting).

In [ ]:
# ── Setup: clone repo, verify GPU, enable CUDA optimizations ──
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

# CUDA optimizations
torch.set_float32_matmul_precision('high')  # enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}')

In [ ]:
# ── Download data ──
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
# ── Imports ──
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import CUM5v6, CUM8v1, CUM9v1
print('Imports OK')

In [ ]:
# ── Config (proven: no overfitting on TinyShakespeare) ──
MODEL_CFG = dict(
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    ctx_len=256,
)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

# Sanity check
test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
n_params = sum(p.numel() for p in test_model.parameters())
n_hidden = sum(p.numel() for p in test_model.get_hidden_2d_params())
hidden_shapes = [(p.shape[0], p.shape[1]) for p in test_model.get_hidden_2d_params()]
print(f'Model: {n_params/1e6:.1f}M params ({n_hidden/1e6:.1f}M hidden 2D)')
print(f'Hidden matrix shapes: {set(hidden_shapes)}')
print(f'Batch={BATCH_SIZE}, Steps={TOTAL_STEPS}')
del test_model

In [ ]:
# ── Data & Training Infrastructure ──

class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
# ── Training Loop ──

def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    # Warmup torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    # Final eval (more samples)
    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
# ── Optimizer Factory ──

def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    t = cfg['type']

    if t == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
    elif t == '5v6':
        main_opt = CUM5v6(
            hidden_2d, lr=BASE_LR,
            mode=cfg.get('mode', 'ns_blend'),
            ns_save_at=cfg.get('ns_save_at', 2),
            ns_blend=cfg.get('ns_blend', 0.25),
        )
    elif t == '8v1':
        main_opt = CUM8v1(
            hidden_2d, lr=BASE_LR,
            method=cfg.get('method', 'matrix'),
            mode=cfg.get('mode', 'two_point'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            save_at_a=cfg.get('save_at_a', 1),
            save_at_b=cfg.get('save_at_b', 3),
            blend_a=cfg.get('blend_a', 0.05),
            blend_b=cfg.get('blend_b', 0.10),
            sv_blend_mode=cfg.get('sv_blend_mode', 'arithmetic'),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
            alpha_curv=cfg.get('alpha_curv', 0.15),
            beta_temp=cfg.get('beta_temp', 0.15),
            total_steps=TOTAL_STEPS,
        )
    elif t == '9v1':
        main_opt = CUM9v1(
            hidden_2d, lr=BASE_LR,
            beta_fast=cfg.get('beta_fast', 0.80),
            beta_slow=cfg.get('beta_slow', 0.95),
            blend=cfg.get('blend', 0.15),
            mode=cfg.get('mode', 'pre_ns'),
            ns_steps=5,
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main_opt, aux_opt


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, main_opt, aux_opt, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='8a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    v56 = next((r['val_loss'] for r in results if r['type'] == '5v6'), None)

    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon | vs 5v6 | Time |')
    print(f'|--------|----------|---------|--------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r['error']:
            print(f'| {r["name"]} | FAILED | — | — | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '—'
        v5 = f'{r["val_loss"] - v56:+.4f}' if v56 else '—'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {v5} | {r["elapsed"]:.0f}s |')

    new = [r for r in sorted(results, key=lambda x: x['val_loss'])
           if r['type'] not in ('Muon', '5v6') and not r['error']]
    if new:
        b = new[0]
        print(f'\nBEST NEW: {b["name"]} -> {b["val_loss"]:.4f}', end='')
        if muon: print(f' (vs Muon: {b["val_loss"]-muon:+.4f})', end='')
        if v56: print(f' (vs 5v6: {b["val_loss"]-v56:+.4f})', end='')
        print()


print('Factory ready')

---
## Series 8a: Two-Point Blend Exploration

**Q1:** Does extending NS past 5 steps help the blend? (NS₂+NS₈ vs NS₂+NS₅)

**Q2:** Does a different save point carry better curvature? (NS₃ vs NS₂)

In [ ]:
CONFIGS_8A = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 s2 b=0.25',       'type': '5v6', 'ns_save_at': 2, 'ns_blend': 0.25},
    # Extended NS: 8 total steps, save at 2, blend 0.15
    {'name': '8v1 s2+NS8 b=0.15',   'type': '8v1', 'method': 'matrix', 'mode': 'two_point',
     'ns_steps': 8, 'save_at': 2, 'blend': 0.15},
    # Different save point: step 3 (more denoised), 7 total steps
    {'name': '8v1 s3+NS7 b=0.15',   'type': '8v1', 'method': 'matrix', 'mode': 'two_point',
     'ns_steps': 7, 'save_at': 3, 'blend': 0.15},
]

print(f'{len(CONFIGS_8A)} configs')

In [ ]:
results_8a = run_all(CONFIGS_8A)
show_results(results_8a, '8a')

---
## Series 8b: Three-Point + SV-Space Blend

**Q3:** Does adding a third interpolation point (NS₁+NS₃+NS₅) improve over two-point?

**Q4:** Does geometric mean of SVs beat arithmetic mean?

In [ ]:
CONFIGS_8B = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 s2 b=0.25',       'type': '5v6', 'ns_save_at': 2, 'ns_blend': 0.25},
    # Three-point blend: NS1 + NS3 + NS5
    {'name': '8v1 3pt 1+3+5',       'type': '8v1', 'method': 'matrix', 'mode': 'three_point',
     'ns_steps': 5, 'save_at_a': 1, 'save_at_b': 3, 'blend_a': 0.05, 'blend_b': 0.10},
    # Geometric SV blend (SVD path)
    {'name': '8v1 geom s2 b=0.15',  'type': '8v1', 'method': 'svd', 'mode': 'sv_blend',
     'ns_steps': 5, 'save_at': 2, 'blend': 0.15, 'sv_blend_mode': 'geometric'},
]

print(f'{len(CONFIGS_8B)} configs')

In [ ]:
results_8b = run_all(CONFIGS_8B)
show_results(results_8b, '8b')

---
## Series 9a: Dual-Momentum + Input Blend

**Q5:** Does blending two momentum time-scales before NS help? (generalization test)

**Q6:** Does temporal averaging of NS outputs (denoised momentum EMA) help?

In [ ]:
CONFIGS_9A = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '5v6 s2 b=0.25',       'type': '5v6', 'ns_save_at': 2, 'ns_blend': 0.25},
    # Dual-momentum: fast (beta=0.80) + slow (beta=0.95), blend before NS
    {'name': '9v1 pre bf=.80 bs=.95', 'type': '9v1', 'mode': 'pre_ns',
     'beta_fast': 0.80, 'beta_slow': 0.95, 'blend': 0.15},
    # Input blend: EMA of past NS outputs blended into current
    {'name': '8v1 input-blend',      'type': '8v1', 'method': 'matrix', 'mode': 'input_blend',
     'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
]

print(f'{len(CONFIGS_9A)} configs')

In [ ]:
results_9a = run_all(CONFIGS_9A)
show_results(results_9a, '9a')

---
## Series 8c: Combined + Scheduled Three-Point

**Q7:** Does combining iterate blend (within-step) + input blend (across-step) beat either alone?

**Q8:** Does scheduling NS₁ weight (high→0) give three-point's early speed with two-point's clean finish?

In [ ]:
CONFIGS_8C = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '8v1 input-blend',      'type': '8v1', 'method': 'matrix', 'mode': 'input_blend',
     'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    # Combined: iterate blend (s2, b=0.15) + input blend (beta=0.5, alpha=0.15)
    {'name': '8v1 combined',         'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    # Scheduled three-point: NS1 weight 0.05→0, NS3 weight stays 0.10
    {'name': '8v1 sched-3pt',        'type': '8v1', 'method': 'matrix', 'mode': 'scheduled_three',
     'save_at_a': 1, 'save_at_b': 3, 'blend_a': 0.05, 'blend_b': 0.10},
]

print(f'{len(CONFIGS_8C)} configs')

In [ ]:
results_8c = run_all(CONFIGS_8C)
show_results(results_8c, '8c')

---
## Series 8d: Intelligent Blending — Adaptive Residual + Cosine Gating

**Q9:** Does adaptive weighting (trust corrections proportional to magnitude) beat fixed weights?

**Q10:** Does cosine gating (only inject when curvature & temporal signals agree) improve quality?

Both use residual decomposition: `update = NS₅ + α*δ_curv + β*δ_temp` where corrections are weighted intelligently.

In [ ]:
CONFIGS_8D = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    {'name': '8v1 input-blend',      'type': '8v1', 'method': 'matrix', 'mode': 'input_blend',
     'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    # Adaptive residual: corrections weighted by their magnitude
    {'name': '8v1 adaptive-res',     'type': '8v1', 'method': 'matrix', 'mode': 'adaptive_residual',
     'save_at': 2, 'alpha_curv': 0.15, 'beta_temp': 0.15, 'input_blend_beta': 0.5},
    # Cosine-gated: corrections only applied when curvature & temporal agree
    {'name': '8v1 cos-gated',        'type': '8v1', 'method': 'matrix', 'mode': 'cosine_gated',
     'save_at': 2, 'alpha_curv': 0.15, 'beta_temp': 0.15, 'input_blend_beta': 0.5},
]

print(f'{len(CONFIGS_8D)} configs')

In [ ]:
results_8d = run_all(CONFIGS_8D)
show_results(results_8d, '8d')

---
## Series 8e: Follow-up — refine best from 8c/8d

In [ ]:
# TODO: fill in based on 8c/8d results
CONFIGS_8E = [
    {'name': 'Muon NS=5',           'type': 'Muon'},
    # Add refinements of best from 8c/8d
]

if len(CONFIGS_8E) > 1:
    results_8e = run_all(CONFIGS_8E)
    show_results(results_8e, '8e')
else:
    print('Fill in configs based on 8c/8d results, then re-run')